# CS781 Capstone Phase 5: Legal Statute Identification (LSI)
## Submission 00: Frozen Backbone Linear Probe & Feature Representation Audit
### Architectural Profile: Model 4 Peak Capacity Boundary Decoder (153.9M Parameters)
* **Backbone:** 14 layers, d=896, H_q=14, H_kv=7 (100% FROZEN)
* **Role:** Unconfounded baseline evaluating raw Phase 1 pre-trained legal representation quality
* **Optimization Shortcut:** One-time offline caching of chunk features to `sub00_frozen_chunk_cache.pt` (<2 min head training)
* **Classifier:** Label-Wise Chunk Attention (CAML) + StatuteClassifierHead + Asymmetric Loss (ASL)
* **Evaluation:** Split-half threshold calibration (dev_tune -> dev_eval), Macro-F1 primary metric


In [ ]:
# 1. Environment & Self-Contained Phase 5 Core Library (100% Standalone)
import os
import sys
import gc
import json
import time
import math
import random
import zipfile
import shutil
import base64
import zlib
import types
import threading
import numpy as np
import matplotlib.pyplot as plt
from typing import Dict, List, Tuple, Any, Optional, Union

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ==============================================================================
# SELF-CONTAINED PHASE 5 CORE MODULE INLINING (Zero external file dependencies)
# ==============================================================================
_PHASE5_CORE_B64 = '...'

try:
    _core_src = zlib.decompress(base64.b64decode(_PHASE5_CORE_B64)).decode("utf-8")
    # 1. Write phase5_common.py directly to working directory so any subprocess or script can access it
    with open("phase5_common.py", "w", encoding="utf-8") as _f:
        _f.write(_core_src)
    
    # 2. Execute directly into an in-memory module
    p5 = types.ModuleType("phase5_common")
    exec(_core_src, p5.__dict__)
    sys.modules["phase5_common"] = p5
    print("Self-contained phase5_common core initialized successfully (100% standalone).")
except Exception as _e:
    print(f"Warning during self-contained core unpacking: {_e}")
    import phase5_common as p5

# Secure dynamic token retrieval (never hardcoded)
HF_TOKEN = p5.get_hf_token()
if HF_TOKEN:
    print("Hugging Face Hub authentication token identified successfully.")
else:
    print("WARNING: HF_TOKEN not found. Remote Hub sync will be local only.")


In [ ]:
# 2. Compute Device Configuration & Reproducibility
device = p5.get_best_device()
print(f"Active Compute Device: {device}")

p5.seed_everything(42)
print("Deterministic random seeds initialized (Seed 42).")


In [ ]:
# 3. Experiment Configuration & Hyperparameters
# =====================================================================
# ⚙️ USER RUN CONTROL PANEL
# =====================================================================
# 1. PROFILE: Choose run mode:
#      - "core"  : Full production training on 100% of data (Official Phase 5 Run)
#      - "smoke" : Rapid end-to-end verification (<2 min on ~16-32 samples)
#                  Verifies model loading, forward, backward, calibration & zip packaging
PROFILE = "core"  # Options: "core" | "smoke"

# 2. EPOCHS: Number of training epochs (1 for smoke test; 5 for core production)
EPOCHS = 1 if PROFILE == "smoke" else 5

# 3. HUB_SYNC_STEP_INTERVAL: Checkpoint sync frequency to HF Hub (in optimizer steps)
HUB_SYNC_STEP_INTERVAL = 200
PAD_TOKEN_ID = p5.PAD_TOKEN_ID
# =====================================================================

EXPERIMENT_NAME = "sub00_baseline_linear_probe"
MODEL_KEY = "model4"  # Model 4: Peak-Capacity Boundary Decoder (153.9M)
SUBFOLDER = "model4-capacity-boundary-154m"
TARGET_METRIC = "Macro-F1 (Official Primary Metric)"

# Output paths
OUTPUT_DIR = os.path.abspath(f"./checkpoint_{EXPERIMENT_NAME}")
CACHE_FILE = os.path.join(OUTPUT_DIR, "sub00_frozen_chunk_cache_smoke.pt" if PROFILE == "smoke" else "sub00_frozen_chunk_cache.pt")
SUBMISSION_ZIP = os.path.join(OUTPUT_DIR, "submission.zip")
PREDICTION_JSON = os.path.join(OUTPUT_DIR, "prediction.json")
METRICS_JSON = os.path.join(OUTPUT_DIR, "val_metrics.json")
LOSS_PLOT_PATH = os.path.join(OUTPUT_DIR, "training_loss.png")

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Training Hyperparameters
BATCH_SIZE = 16
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 0.01
D_ATTN = 128
CHUNK_WINDOW = 512
CHUNK_OVERLAP = 64
CHUNK_STRIDE = 448
TRAIN_SAMPLE_K = 6

print(f"Configuration set for: {EXPERIMENT_NAME}")
print(f"Run Profile: {PROFILE.upper()} | Configured Epochs: {EPOCHS} | Sync Interval: {HUB_SYNC_STEP_INTERVAL} steps")
print(f"Output directory: {OUTPUT_DIR}")


In [ ]:
# 4. Ingest Phase 5 Dataset & Setup Tokenizer
print("Ingesting Phase 5 LSI dataset...")
train_records, dev_tune, dev_eval, test_records, statute2idx, idx2statute = p5.ingest_phase5_dataset(HF_TOKEN)

# Handle SMOKE test profile slicing
if PROFILE == "smoke":
    print("=" * 65)
    print(">>> [PROFILE: SMOKE] Slicing dataset for rapid end-to-end verification <<<")
    print(f">>> Configured EPOCHS: {EPOCHS} | Hub Sync Interval: {HUB_SYNC_STEP_INTERVAL} steps")
    print("=" * 65)
    train_records = train_records[:32]
    dev_tune = dev_tune[:16]
    dev_eval = dev_eval[:16]
    test_records = test_records[:16]
    if EPOCHS > 2:
        print(f"[Smoke Mode Notice] Capping EPOCHS from {EPOCHS} to 1 for quick execution.")
        EPOCHS = 1

NUM_LABELS = len(statute2idx)
print(f"Total Unique Statutes (|L|): {NUM_LABELS}")
print(f"Train documents: {len(train_records):,}")
print(f"Dev Tune documents: {len(dev_tune):,}")
print(f"Dev Eval documents: {len(dev_eval):,}")
print(f"Test documents: {len(test_records):,}")

# Load custom scratch-trained 32K legal tokenizer
tokenizer = p5.load_legal_tokenizer(p5.HF_TOKENIZER_REPO, token=HF_TOKEN)
p5.PAD_TOKEN_ID = tokenizer.pad_token_id
PAD_TOKEN_ID = tokenizer.pad_token_id
if tokenizer.eos_token_id is not None:
    p5.EOS_TOKEN_ID = tokenizer.eos_token_id
    EOS_TOKEN_ID = tokenizer.eos_token_id
print(f"Loaded Tokenizer: Vocab={len(tokenizer):,}, Pad={tokenizer.pad_token_id}, EOS={tokenizer.eos_token_id}")


In [ ]:
# 5. Strict Pretrained Backbone Loading (Assert Zero Missing Keys)
print(f"Loading Model 4 backbone from {p5.HF_MASTER_REPO}/{SUBFOLDER}...")
backbone, config_dict = p5.load_strict_phase1_backbone(MODEL_KEY, hf_token=HF_TOKEN)

backbone = backbone.to(device)
backbone.eval()

# STRICT UNCONFOUNDED BASELINE REQUIREMENT:
# Freeze 100% of the backbone parameters
for param in backbone.parameters():
    param.requires_grad = False

total_backbone_params = sum(p.numel() for p in backbone.parameters())
trainable_backbone_params = sum(p.numel() for p in backbone.parameters() if p.requires_grad)

print(f"Model 4 Total Backbone Parameters: {total_backbone_params / 1e6:.2f}M")
print(f"Model 4 Trainable Backbone Parameters: {trainable_backbone_params} (100% FROZEN)")
assert trainable_backbone_params == 0, "Backbone must be 100% frozen for Sub 00!"


In [ ]:
# 6. One-Time Offline Chunk Feature Extraction & Disk Caching
# Optimizes training: extracts [B, M, 2*d_model] chunk representations once.
# Subsequent training completes in < 2 minutes directly on cached representations.

def extract_document_features(doc_record, is_train=False):
    tokens = tokenizer.encode(doc_record["text"], add_special_tokens=False) + [tokenizer.eos_token_id]
    input_ids, attn_mask = p5.chunk_document_tokens(tokens, window_size=CHUNK_WINDOW, stride=CHUNK_STRIDE)
    
    if is_train:
        input_ids, attn_mask = p5.sample_training_chunks(input_ids, attn_mask, doc_record["text"], tokenizer, k=TRAIN_SAMPLE_K)
    
    # Run through backbone in no_grad mode
    with torch.no_grad():
        inp = input_ids.to(device)
        mask = attn_mask.to(device)
        outputs = backbone(input_ids=inp, attention_mask=mask, output_hidden_states=True, return_dict=True)
        top4 = list(outputs.hidden_states[-4:])
        # Simple uniform mean mixture for frozen representation audit
        mixed = sum(top4) / 4.0
        chunk_feats = p5.extract_chunk_features(mixed, mask).cpu()
    
    # Build binary label vector
    label_vec = torch.zeros(NUM_LABELS, dtype=torch.float32)
    for lbl in doc_record.get("labels", []):
        if int(lbl) in statute2idx:
            label_vec[statute2idx[int(lbl)]] = 1.0
            
    return chunk_feats, label_vec, doc_record["id"]

if os.path.exists(CACHE_FILE):
    print(f"Found existing cached representation file: {CACHE_FILE}")
    cached_data = torch.load(CACHE_FILE, map_location="cpu")
    print("Loaded cached document features successfully.")
else:
    print(f"Computing representations and caching to: {CACHE_FILE}...")
    start_time = time.time()
    
    cached_data = {"train": [], "dev_tune": [], "dev_eval": [], "test": []}
    
    for split_name, dataset_records, is_tr in [
        ("train", train_records, True),
        ("dev_tune", dev_tune, False),
        ("dev_eval", dev_eval, False),
        ("test", test_records, False)
    ]:
        print(f"Caching features for split: {split_name} ({len(dataset_records)} docs)...")
        for i, rec in enumerate(dataset_records):
            feats, targets, doc_id = extract_document_features(rec, is_train=is_tr)
            cached_data[split_name].append({
                "feats": feats,  # [M, 2*d_model]
                "targets": targets,  # [NUM_LABELS]
                "id": doc_id
            })
            if (i + 1) % 500 == 0 or (i + 1) == len(dataset_records):
                print(f"  Processed {i + 1}/{len(dataset_records)} docs...")
                
    torch.save(cached_data, CACHE_FILE)
    elapsed = time.time() - start_time
    print(f"Caching completed and saved to {CACHE_FILE} in {elapsed / 60:.2f} minutes.")

# Free backbone from VRAM to preserve memory for training
del backbone
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()


In [ ]:
# 7. PyTorch Dataset & Dynamic Collation for Variable Chunks
class CachedLSIDataset(Dataset):
    def __init__(self, data_list):
        self.data = data_list

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        return item["feats"], item["targets"], item["id"]

def collate_cached_batch(batch):
    # Pad variable chunk count M across documents in the batch
    max_m = max(item[0].size(0) for item in batch)
    d_feat = batch[0][0].size(1)
    
    b_size = len(batch)
    padded_feats = torch.zeros(b_size, max_m, d_feat, dtype=torch.float32)
    chunk_mask = torch.zeros(b_size, max_m, dtype=torch.float32)
    targets = torch.stack([item[1] for item in batch], dim=0)
    doc_ids = [item[2] for item in batch]
    
    for i, item in enumerate(batch):
        m_len = item[0].size(0)
        padded_feats[i, :m_len, :] = item[0]
        valid_chunks = (item[0].abs().sum(dim=-1) > 0).float()
        chunk_mask[i, :m_len] = valid_chunks
        
    return padded_feats, chunk_mask, targets, doc_ids

train_dataset = CachedLSIDataset(cached_data["train"])
dev_tune_dataset = CachedLSIDataset(cached_data["dev_tune"])
dev_eval_dataset = CachedLSIDataset(cached_data["dev_eval"])
test_dataset = CachedLSIDataset(cached_data["test"])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_cached_batch)
dev_tune_loader = DataLoader(dev_tune_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_cached_batch)
dev_eval_loader = DataLoader(dev_eval_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_cached_batch)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_cached_batch)

print("DataLoaders initialized successfully.")


In [ ]:
# 8. Define Trainable Label-Wise Attention & Classifier Head
D_CHUNK_FEAT = 2 * config_dict["hidden_size"]  # 2 * 896 = 1792

class LinearProbeClassifier(nn.Module):
    def __init__(self, d_chunk_feat, num_labels, d_attn=128):
        super().__init__()
        self.attn_pool = p5.LabelWiseChunkAttention(d_chunk_feat, num_labels, d_attn=d_attn)
        self.head = p5.StatuteClassifierHead(d_chunk_feat, num_labels)

    def forward(self, chunk_feats, chunk_mask=None):
        statute_reps = self.attn_pool(chunk_feats, chunk_mask=chunk_mask)
        logits = self.head(statute_reps)
        return logits

probe_model = LinearProbeClassifier(D_CHUNK_FEAT, NUM_LABELS, d_attn=D_ATTN).to(device)

total_trainable = sum(p.numel() for p in probe_model.parameters() if p.requires_grad)
print(f"Linear Probe Trainable Parameters: {total_trainable:,}")

criterion = p5.AsymmetricLoss(gamma_neg=4.0, gamma_pos=0.0, clip_margin=0.05)
optimizer = torch.optim.AdamW(probe_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

hub_sync = p5.ResilientHubSyncManager(
    repo_id=p5.HF_MASTER_REPO,
    subfolder=f"Phase_5/{EXPERIMENT_NAME}",
    token=HF_TOKEN,
    is_peft=True
)
print("Optimizer, ASL criterion, and ResilientHubSyncManager ready.")


In [ ]:
# 9. Rapid Training Directly on Cached Features (< 2 Minutes)
print(f"Beginning training for {EPOCHS} epochs on cached representations...")
history = {"train_loss": [], "dev_eval_f1": []}
best_eval_f1 = -1.0

start_train_time = time.time()

global_step = 0
for epoch in range(1, EPOCHS + 1):
    probe_model.train()
    total_loss = 0.0
    num_batches = 0
    
    for batch_feats, batch_mask, batch_targets, _ in train_loader:
        batch_feats = batch_feats.to(device)
        batch_mask = batch_mask.to(device)
        batch_targets = batch_targets.to(device)
        
        optimizer.zero_grad()
        logits = probe_model(batch_feats, chunk_mask=batch_mask)
        loss = criterion(logits, batch_targets)
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(probe_model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item()
        num_batches += 1
        global_step += 1
        
        # Sync resume state to Hub every HUB_SYNC_STEP_INTERVAL (200) steps or on final smoke step
        if (global_step % HUB_SYNC_STEP_INTERVAL == 0) or (PROFILE == "smoke" and num_batches == len(train_loader) and epoch == EPOCHS):
            print(f"[HubSync] Syncing checkpoint state at step {global_step}...")
            hub_sync.save_and_sync_step(
                step=global_step,
                epoch=epoch,
                model=probe_model,
                optimizer=optimizer,
                scheduler=scheduler,
                local_dir=OUTPUT_DIR,
                is_best=False
            )
        
    scheduler.step()
    avg_train_loss = total_loss / max(1, num_batches)
    history["train_loss"].append(avg_train_loss)
    
    # Evaluate on dev_tune split to select optimal checkpoint (prevents selection leak on dev_eval)
    probe_model.eval()
    tune_probs_list, tune_targets_list = [], []
    with torch.no_grad():
        for b_feats, b_mask, b_targets, _ in dev_tune_loader:
            b_feats = b_feats.to(device)
            b_mask = b_mask.to(device)
            l = probe_model(b_feats, chunk_mask=b_mask)
            p = torch.sigmoid(l).cpu().numpy()
            tune_probs_list.append(p)
            tune_targets_list.append(b_targets.numpy())
            
    tune_probs = np.vstack(tune_probs_list)
    tune_targets = np.vstack(tune_targets_list)
    temp_preds = (tune_probs >= 0.30).astype(np.int32)
    # Ensure top-1 fallback
    empty_mask = temp_preds.sum(axis=1) == 0
    temp_preds[empty_mask, tune_probs[empty_mask].argmax(axis=1)] = 1
    
    curr_tune_f1 = p5.compute_lsi_metrics(tune_targets, temp_preds)["macro_f1"]
    history["dev_eval_f1"].append(curr_tune_f1)
    
    is_best = curr_tune_f1 > best_eval_f1
    if is_best:
        best_eval_f1 = curr_tune_f1
        best_probe_state = {k: v.detach().cpu().clone() for k, v in probe_model.state_dict().items()}
        torch.save(best_probe_state, os.path.join(OUTPUT_DIR, "best_probe_weights.pt"))
        
    print(f"Epoch [{epoch}/{EPOCHS}] | Train Loss: {avg_train_loss:.4f} | Dev Tune Macro-F1: {curr_tune_f1:.4f} {'(New Best!)' if is_best else ''}")
    
    # Asynchronously sync checkpoint
    hub_sync.save_and_sync_step(
        step=epoch,
        epoch=epoch,
        model=probe_model,
        optimizer=optimizer,
        scheduler=scheduler,
        local_dir=OUTPUT_DIR,
        val_macro_f1=curr_tune_f1,
        is_best=is_best
    )

train_duration = time.time() - start_train_time
print(f"Training completed successfully in: {train_duration:.1f} seconds ({train_duration/60:.2f} minutes).")
print(f"Peak Dev Tune Macro-F1: {best_eval_f1:.4f}")


In [ ]:
# 10. Training Loss Curve & Performance Visualization
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(range(1, EPOCHS + 1), history["train_loss"], marker='o', color='navy', label="Train ASL Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title(f"{EXPERIMENT_NAME} - Loss Curve")
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(range(1, EPOCHS + 1), history["dev_eval_f1"], marker='s', color='darkgreen', label="Dev Eval Macro-F1")
plt.xlabel("Epoch")
plt.ylabel("Macro-F1")
plt.title("Validation Trajectory")
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()

plt.tight_layout()
plt.savefig(LOSS_PLOT_PATH, dpi=200)
plt.show()
print(f"Saved performance plot to: {LOSS_PLOT_PATH}")


In [ ]:
# 11. Split-Half Calibration on dev_tune & Unbiased Evaluation on dev_eval
# Restore best checkpoint snapshot before calibration
if 'best_probe_state' in locals() and best_probe_state is not None:
    probe_model.load_state_dict(best_probe_state)
    print("Successfully restored best probe weights from in-memory snapshot before calibration.")
elif os.path.exists(os.path.join(OUTPUT_DIR, "best_probe_weights.pt")):
    probe_model.load_state_dict(torch.load(os.path.join(OUTPUT_DIR, "best_probe_weights.pt"), map_location=device))
    print("Restored best probe weights from disk.")

probe_model.eval()

def gather_split_predictions(loader):
    probs_list, targets_list, ids_list = [], [], []
    with torch.no_grad():
        for b_feats, b_mask, b_targets, b_ids in loader:
            b_feats = b_feats.to(device)
            b_mask = b_mask.to(device)
            logits = probe_model(b_feats, chunk_mask=b_mask)
            probs = torch.sigmoid(logits).cpu().numpy()
            probs_list.append(probs)
            targets_list.append(b_targets.numpy())
            ids_list.extend(b_ids)
    return np.vstack(probs_list), np.vstack(targets_list), ids_list

print("Collecting predictions on dev_tune and dev_eval splits...")
tune_probs, tune_targets, tune_ids = gather_split_predictions(dev_tune_loader)
eval_probs, eval_targets, eval_ids = gather_split_predictions(dev_eval_loader)

# Fit Calibrator strictly on dev_tune
calibrator = p5.SplitHalfCalibrator(statute2idx, idx2statute)
train_label_support = np.array([item["targets"].numpy() for item in cached_data["train"]]).sum(axis=0)
calibrator.fit(tune_targets, tune_probs, train_support=train_label_support)

# Apply calibrated thresholds to evaluate dev_eval (strictly unbiased reporting)
eval_binary_preds = calibrator.predict(eval_probs)
unbiased_metrics = p5.compute_lsi_metrics(eval_targets, eval_binary_preds)

print("\n" + "=" * 60)
print("UNBIASED DEV_EVAL VALIDATION PERFORMANCE (SUB 00)")
print("=" * 60)
for k, v in unbiased_metrics.items():
    print(f"  {k:20s}: {v:.4f}")
print("=" * 60)

with open(METRICS_JSON, "w", encoding="utf-8") as f:
    json.dump(unbiased_metrics, f, indent=2)
print(f"Saved unbiased validation metrics to: {METRICS_JSON}")


In [ ]:
# 12. Test Set Inference & Strict Codabench Packaging
print("Generating test predictions on official test set...")
test_probs_list, test_ids_list = [], []

with torch.no_grad():
    for b_feats, b_mask, _, b_ids in test_loader:
        b_feats = b_feats.to(device)
        b_mask = b_mask.to(device)
        logits = probe_model(b_feats, chunk_mask=b_mask)
        probs = torch.sigmoid(logits).cpu().numpy()
        test_probs_list.append(probs)
        test_ids_list.extend(b_ids)

test_probs = np.vstack(test_probs_list)
print(f"Test probability matrix generated: {test_probs.shape}")

# Save raw test probabilities for Sub 07 ensemble
# Save both dev_probs.npy (tune + eval) and test probabilities for Sub 07 ensemble
dev_probs_combined = np.vstack([tune_probs, eval_probs])
np.save(os.path.join(OUTPUT_DIR, "dev_probs.npy"), dev_probs_combined)
np.save(os.path.join(OUTPUT_DIR, "sub00_test_probs.npy"), test_probs)
np.save(os.path.join(OUTPUT_DIR, "dev_probs.npy"), eval_probs)

# Format JSON predictions
test_predictions = calibrator.format_predictions_json(test_probs, test_ids_list)

# Validate and package into submission.zip
p5.package_codabench_submission(
    predictions_list=test_predictions,
    output_zip_path=SUBMISSION_ZIP,
    expected_doc_ids=test_ids_list
)
print("Codabench submission package verified and ready.")


In [ ]:
# 13. Atomically Push Sub 00 Artifacts to Master Hugging Face Hub
if HF_TOKEN:
    print(f"Uploading Sub 00 deliverables to {p5.HF_MASTER_REPO}/Phase_5/{EXPERIMENT_NAME}...")
    try:
        api = p5.HfApi(token=HF_TOKEN)
        # Upload submission archive
        api.upload_file(
            path_or_fileobj=SUBMISSION_ZIP,
            path_in_repo=f"Phase_5/{EXPERIMENT_NAME}/submission.zip",
            repo_id=p5.HF_MASTER_REPO,
            commit_message="Add Sub 00 Codabench submission.zip"
        )
        # Upload prediction.json
        api.upload_file(
            path_or_fileobj=PREDICTION_JSON,
            path_in_repo=f"Phase_5/{EXPERIMENT_NAME}/prediction.json",
            repo_id=p5.HF_MASTER_REPO,
            commit_message="Add Sub 00 prediction.json"
        )
        # Upload validation metrics
        api.upload_file(
            path_or_fileobj=METRICS_JSON,
            path_in_repo=f"Phase_5/{EXPERIMENT_NAME}/val_metrics.json",
            repo_id=p5.HF_MASTER_REPO,
            commit_message="Add Sub 00 val_metrics.json"
        )
        # Upload loss curve
        if os.path.exists(LOSS_PLOT_PATH):
            api.upload_file(
                path_or_fileobj=LOSS_PLOT_PATH,
                path_in_repo=f"Phase_5/{EXPERIMENT_NAME}/loss_plot.png",
                repo_id=p5.HF_MASTER_REPO,
                commit_message="Add Sub 00 loss plot"
            )
        # Upload probability arrays for Sub 07 ensemble
        dev_prob_p = os.path.join(OUTPUT_DIR, "dev_probs.npy")
        if os.path.exists(dev_prob_p):
            api.upload_file(path_or_fileobj=dev_prob_p, path_in_repo=f"Phase_5/{EXPERIMENT_NAME}/dev_probs.npy", repo_id=p5.HF_MASTER_REPO, commit_message="Add dev_probs.npy")
        test_prob_p = os.path.join(OUTPUT_DIR, "sub00_test_probs.npy")
        if os.path.exists(test_prob_p):
            api.upload_file(path_or_fileobj=test_prob_p, path_in_repo=f"Phase_5/{EXPERIMENT_NAME}/sub00_test_probs.npy", repo_id=p5.HF_MASTER_REPO, commit_message="Add sub00_test_probs.npy")
        weights_p = os.path.join(OUTPUT_DIR, "best_probe_weights.pt")
        if os.path.exists(weights_p):
            api.upload_file(path_or_fileobj=weights_p, path_in_repo=f"Phase_5/{EXPERIMENT_NAME}/best_probe_weights.pt", repo_id=p5.HF_MASTER_REPO, commit_message="Add best probe weights")
        hub_sync.join()
        print("Sub 00 artifacts uploaded to Hugging Face Hub successfully.")
    except Exception as e:
        print(f"Hub upload notice: {e}. Checkpoint remains safely stored locally in {OUTPUT_DIR}")
else:
    print("HF_TOKEN omitted: All artifacts safely stored locally.")
